# Variance Reduction in Monte Carlo Simulations — corrected code
Camellini Francesco, Kim Youseung, Parietti Alessandro

Corrected version of the Section 7 code (the as-printed transcription is in `variance_reduction_report_code.ipynb`).

Fixes applied:
- **Missing helpers** `gbm_terminal`, `call_payoff`, `put_payoff`, `mc_price_generic_paths` added; `MCResult` / `GBMParams` defined once.
- **7.1.3** `standard_mc_knock_in`: docstring quotes restored (was a SyntaxError); sample variance now uses `ddof=1`.
- **7.3.1** European CV: the control was the call payoff itself (ρ = 1, b̂ = 1, SE ≈ 1e-14). It is now the discounted terminal price S_T, with known mean S0, as described in the report text.
- **7.3.1** `collect_convergence_data_paths`: `B` is only passed to the target payoff when given (EU/Asian payoffs take no `B`); ρ̂ and b̂ are now recorded.
- **7.3.2** Asian CV: added the geometric-average Asian call as a control (closed-form price), alongside the EU-call control.
- **7.3.3** Knock-in CV: the control's true price is the Black–Scholes call price (`bs_call_price`).
- **7.4.2** `asian_IS`: undefined `se` → `std_err`, stray `)` removed (was a SyntaxError).
- **7.4.3** Knock-in IS: `np.random.seed` outside a numba function does not seed numba's generator, so a `seed` argument now seeds it inside; variance uses `ddof=1`.

## 0. Shared helpers

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from scipy.stats import norm
from scipy.optimize import brentq


@dataclass
class MCResult:
    price: float
    std_error: float
    variance: float
    ci95: tuple[float, float]
    n_paths: int


@dataclass
class GBMParams:
    S0: float = 100.0
    K: float = 100.0
    r: float = 0.03
    sigma: float = 0.20
    T: float = 1.0
    n: int = 100_000
    seed: int = 7


def gbm_terminal(S0, r, sigma, T, n_paths, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    Z = rng.standard_normal(n_paths)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    return ST, Z


def call_payoff(ST, K):
    return np.maximum(ST - K, 0.0)


def put_payoff(ST, K):
    return np.maximum(K - ST, 0.0)


def bs_call_price(S0, K, r, sigma, T):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def bs_put_price(S0, K, r, sigma, T):
    return bs_call_price(S0, K, r, sigma, T) - S0 + K * np.exp(-r * T)


# GBM paths (monitoring dates t_1..t_n, S0 not included)
def gbm_paths(params: GBMParams, n_steps: int = 1) -> np.ndarray:
    rng = np.random.default_rng(params.seed)
    dt = params.T / n_steps
    Z = rng.standard_normal((params.n, n_steps))
    drift = (params.r - 0.5 * params.sigma**2) * dt
    diffusion = params.sigma * np.sqrt(dt) * Z
    log_S = np.log(params.S0) + np.cumsum(drift + diffusion, axis=1)
    return np.exp(log_S)  # shape: (n, n_steps)


# Generic plain MC on paths
def mc_price_generic_paths(params: GBMParams, payoff_func, n_steps: int = 1):
    S_paths = gbm_paths(params, n_steps=n_steps)
    disc = np.exp(-params.r * params.T)
    Y = disc * payoff_func(S_paths, params)
    mu = float(np.mean(Y))
    var = float(np.var(Y, ddof=1))
    se = float(np.sqrt(var / params.n))
    ci = (mu - 1.96 * se, mu + 1.96 * se)
    return mu, var, se, ci

## 7.1 Standard Monte Carlo
### 7.1.1 European Options

In [15]:
def mc_euro_bs_plain(S0: float, K: float, r: float, sigma: float, T: float,
                     n_paths: int = 100_000, option: str = "call", rng=None) -> MCResult:
    """
    Plain MC estimator for a European option under Black-Scholes.
    Returns price, std error, variance and 95% CI.
    """
    ST, _ = gbm_terminal(S0, r, sigma, T, n_paths, rng=rng)
    if option == "call":
        payoffs = call_payoff(ST, K)
    elif option == "put":
        payoffs = put_payoff(ST, K)
    else:
        raise ValueError("option must be 'call' or 'put'")

    disc_payoffs = np.exp(-r * T) * payoffs
    # Unbiased sample variance (ddof=1) to estimate Var estimator
    variance = np.var(disc_payoffs, ddof=1)
    mean = np.mean(disc_payoffs)
    se = np.sqrt(variance / n_paths)
    ci = (mean - 1.96 * se, mean + 1.96 * se)
    return MCResult(price=mean, std_error=se, variance=variance, ci95=ci, n_paths=n_paths)

### 7.1.2 Asian Options

In [16]:
def asian_arith_plain(
    S0: float, K: float, r: float, sigma: float, T: float,
    n_steps: int = 252, n_paths: int = 100_000,
    option: str = "call", include_S0: bool = True, rng=None
):
    """
    Plain Monte Carlo for arithmetic-average Asian option under GBM.
    Returns MCResult(price, std_error, variance, ci95, n_paths) same fields as European.

    Average definition:
      if include_S0=True : A = (1/(n_steps+1)) * sum_{j=0}^{n_steps} S(t_j)
      if include_S0=False: A = (1/n_steps) * sum_{j=1}^{n_steps} S(t_j)
    """
    if rng is None:
        rng = np.random.default_rng()

    dt = T / n_steps
    nudt = (r - 0.5 * sigma**2) * dt
    sdt = sigma * np.sqrt(dt)

    # Generate normal distribution for all paths and steps
    Z = rng.standard_normal(size=(n_paths, n_steps))

    # Simulate in log-space
    log_increments = nudt + sdt * Z
    log_paths = np.cumsum(log_increments, axis=1)  # shape: (n_paths, n_steps)

    # Build price paths; prepend S0 if the average includes it
    if include_S0:
        S = np.concatenate([np.full((n_paths, 1), S0), S0 * np.exp(log_paths)], axis=1)
        A = S.mean(axis=1)  # average over n_steps+1 points
    else:
        S = S0 * np.exp(log_paths)
        A = S.mean(axis=1)  # average over n_steps points

    # Payoff
    if option == "call":
        payoffs = np.maximum(A - K, 0.0)
    else:
        payoffs = np.maximum(K - A, 0.0)

    # Discount
    disc_payoffs = np.exp(-r * T) * payoffs

    variance = np.var(disc_payoffs, ddof=1)  # sample variance of discounted payoffs
    mean = np.mean(disc_payoffs)
    se = np.sqrt(variance / n_paths)  # SE of estimator
    ci = (mean - 1.96 * se, mean + 1.96 * se)

    return MCResult(price=mean, std_error=se, variance=variance, ci95=ci, n_paths=n_paths)

### 7.1.3 Digital knock-in Options

In [17]:
def standard_mc_knock_in(S0, K, H, T, r, sigma, m, N):
    """
    Payoff: 10,000 if (min(S_t) <= H) AND (S_T > K), else 0.
    """
    dt = T / m
    sqrt_dt = np.sqrt(dt)
    log_H = np.log(H)

    # Fixed payoff amount
    PAYOFF_AMOUNT = 10000.0

    Z = np.random.standard_normal(size=(N, m))

    # Initialize log-price paths
    # log_S has shape (N, m+1) to include t=0
    log_S = np.zeros((N, m + 1))
    log_S[:, 0] = np.log(S0)

    # Drift per time step
    drift_per_step = (r - 0.5 * sigma ** 2) * dt
    vol_per_step = sigma * sqrt_dt

    # Price evolution (Cumulative sum)
    # log S(t) = log S(0) + cumsum(drift + sigma*Z)
    increments = drift_per_step + vol_per_step * Z
    log_S[:, 1:] = np.log(S0) + np.cumsum(increments, axis=1)

    # 1. Barrier Condition (Down-and-In)
    # Check if the MINimum of the path dropped below log(H)
    min_log_S = np.min(log_S, axis=1)
    hit_barrier = (min_log_S <= log_H)

    # 2. Strike Condition (Call at maturity)
    # Check if the FINAL price is above K
    final_log_S = log_S[:, -1]
    finished_itm = (final_log_S > np.log(K))

    # 3. Joint Event
    # Both conditions must be True
    successful_paths = hit_barrier & finished_itm

    # 4. Payoff Assignment
    raw_payoffs = np.where(successful_paths, PAYOFF_AMOUNT, 0.0)

    # Discounting to present value
    discount_factor = np.exp(-r * T)
    discounted_payoffs = raw_payoffs * discount_factor

    price = np.mean(discounted_payoffs)
    variance = np.var(discounted_payoffs, ddof=1)
    err_std = np.sqrt(variance / N)
    ci = (price - 1.96 * err_std, price + 1.96 * err_std)

    return price, variance

## 7.2 Antithetic Variate
### 7.2.1 European Options

In [18]:
def mc_euro_bs_antithetic(S0: float, K: float, r: float, sigma: float, T: float,
                          n_paths: int = 100_000, option: str = "call", rng=None) -> MCResult:
    if n_paths % 2 != 0:
        raise ValueError("n_paths should be even for antithetic pairing.")

    if rng is None:
        rng = np.random.default_rng()

    half = n_paths // 2
    # Generate half normals and use their negatives as antithetics
    Z = rng.standard_normal(half)
    drift = (r - 0.5 * sigma**2) * T
    volT = sigma * np.sqrt(T)

    # Terminal prices for Z and -Z
    ST_pos = S0 * np.exp(drift + volT * Z)
    ST_neg = S0 * np.exp(drift - volT * Z)

    # Compute payoffs for both and average pairwise (control-style)
    if option == "call":
        g_pos = call_payoff(ST_pos, K)
        g_neg = call_payoff(ST_neg, K)
    elif option == "put":
        g_pos = put_payoff(ST_pos, K)
        g_neg = put_payoff(ST_neg, K)
    else:
        raise ValueError("option must be 'call' or 'put'")

    # Antithetic estimator per pair
    g_hat_pair = 0.5 * (g_pos + g_neg)

    # Discount and aggregate
    disc = np.exp(-r * T)
    disc_payoffs = disc * g_hat_pair

    variance = np.var(disc_payoffs, ddof=1)  # variance of the pair averages
    mean = np.mean(disc_payoffs)
    se = np.sqrt(variance / half)  # SE uses number of *pairs*
    ci = (mean - 1.96 * se, mean + 1.96 * se)

    return MCResult(price=float(mean), std_error=se, variance=float(variance), ci95=ci, n_paths=n_paths)

### 7.2.2 Asian Options

In [19]:
def asian_arith_antithetic(
    S0: float, K: float, r: float, sigma: float, T: float,
    n_steps: int = 252, n_paths: int = 100_000,
    option: str = "call", include_S0: bool = True, rng=None
):
    if n_paths % 2 != 0:
        raise ValueError("n_paths must be even for antithetic pairing.")
    if rng is None:
        rng = np.random.default_rng()

    half = n_paths // 2
    dt = T / n_steps
    nudt = (r - 0.5 * sigma**2) * dt
    sdt = sigma * np.sqrt(dt)

    # Generate normal distribution for half the paths and build antithetic counterparts
    Z = rng.standard_normal(size=(half, n_steps))
    Z_bar = -Z

    # Log-increments for original and antithetic
    log_inc = nudt + sdt * Z
    log_inc_bar = nudt + sdt * Z_bar

    # Accumulate and convert to prices
    log_paths = np.cumsum(log_inc, axis=1)
    log_paths_bar = np.cumsum(log_inc_bar, axis=1)

    if include_S0:
        S = np.concatenate([np.full((half, 1), S0), S0 * np.exp(log_paths)], axis=1)
        S_bar = np.concatenate([np.full((half, 1), S0), S0 * np.exp(log_paths_bar)], axis=1)
        A = S.mean(axis=1)
        A_bar = S_bar.mean(axis=1)
    else:
        S = S0 * np.exp(log_paths)
        S_bar = S0 * np.exp(log_paths_bar)
        A = S.mean(axis=1)
        A_bar = S_bar.mean(axis=1)

    # Payoffs for each pair (original, antithetic)
    if option == "call":
        g = np.maximum(A - K, 0.0)
        g_bar = np.maximum(A_bar - K, 0.0)
    else:
        g = np.maximum(K - A, 0.0)
        g_bar = np.maximum(K - A_bar, 0.0)

    # Pairwise average and discount
    pair_avg = 0.5 * (g + g_bar)
    disc_pair = np.exp(-r * T) * pair_avg  # these are the i.i.d. samples for the estimator

    mean = disc_pair.mean()
    variance = disc_pair.var(ddof=1)  # variance across N/2 pair-averaged samples
    se = np.sqrt(variance / half)  # SE of the estimator uses # pairs
    ci = (mean - 1.96 * se, mean + 1.96 * se)

    return MCResult(price=mean, std_error=se, variance=variance, ci95=ci, n_paths=n_paths)

## 7.3 Control Variates
### 7.3.1 European Options
Control: discounted terminal price $e^{-rT}S_T$, whose expectation is $S_0$ under the risk-neutral measure.

In [20]:
# Control Variate
def mc_price_cv_paths(params: GBMParams, target_payoff_func, control_payoff_func, control_true_price: float, n_steps: int = 1):
    """control_true_price is the known value of E[disc * control_payoff]."""
    S_paths = gbm_paths(params, n_steps=n_steps)
    disc = np.exp(-params.r * params.T)
    payoff_target = target_payoff_func(S_paths, params)
    payoff_control = control_payoff_func(S_paths, params)
    Y = disc * payoff_target
    X = disc * payoff_control
    covXY = float(np.cov(X, Y, ddof=1)[0, 1])
    varX = float(np.var(X, ddof=1))
    varY = float(np.var(Y, ddof=1))
    rho_hat = covXY / (np.sqrt(varX) * np.sqrt(varY)) if varX > 0 and varY > 0 else np.nan
    # b_hat is estimated from the same sample: small O(1/n) bias
    b_hat = covXY / varX if varX > 0 else 0.0
    adj = Y - b_hat * (X - control_true_price)
    mu = float(np.mean(adj))
    var = float(np.var(adj, ddof=1))
    se = float(np.sqrt(var / params.n))
    z = 1.96
    ci = (mu - z * se, mu + z * se)
    return mu, var, se, ci, b_hat, rho_hat

# Payoff functions
def eu_call_payoff_from_paths(paths: np.ndarray, params: GBMParams) -> np.ndarray:
    ST = paths[:, -1]
    return np.maximum(ST - params.K, 0.0)

# Control: terminal price S_T; E[e^{-rT} S_T] = S0
def eu_call_control_from_paths(paths: np.ndarray, params: GBMParams) -> np.ndarray:
    return paths[:, -1]

def eu_call_control_true_price(params: GBMParams) -> float:
    return params.S0

# Convergence data collection
def collect_convergence_data_paths(option_name: str, params_base: GBMParams, payoff_target_func, payoff_control_func, control_true_price: float, n_paths_list, n_steps: int, B: float | None = None):
    """B is passed to the target payoff only when given (barrier options)."""
    if B is None:
        target = payoff_target_func
    else:
        target = lambda paths, params: payoff_target_func(paths, params, B)
    results = {"n": [], "price_mc": [], "var_mc": [], "se_mc": [], "price_cv": [], "var_cv": [], "se_cv": [], "b_hat": [], "rho_hat": []}
    for n in n_paths_list:
        p = GBMParams(S0=params_base.S0, K=params_base.K, r=params_base.r, sigma=params_base.sigma, T=params_base.T, n=int(n), seed=params_base.seed)
        price_mc, var_mc, se_mc, _ = mc_price_generic_paths(p, payoff_func=target, n_steps=n_steps)
        price_cv, var_cv, se_cv, _, b_hat, rho_hat = mc_price_cv_paths(p, target_payoff_func=target, control_payoff_func=payoff_control_func, control_true_price=control_true_price, n_steps=n_steps)
        results["n"].append(int(n))
        results["price_mc"].append(price_mc)
        results["var_mc"].append(var_mc)
        results["se_mc"].append(se_mc)
        results["price_cv"].append(price_cv)
        results["var_cv"].append(var_cv)
        results["se_cv"].append(se_cv)
        results["b_hat"].append(b_hat)
        results["rho_hat"].append(rho_hat)
    return results

### 7.3.2 Asian Options
Target: arithmetic-average Asian call over $t_1,\dots,t_m$ (as produced by `gbm_paths`).

Controls:
- EU call on the same paths (the control used in the report), true price = Black–Scholes.
- Geometric-average Asian call. $\log G \sim N(\mu_G, \sigma_G^2)$ with
  $\mu_G = \log S_0 + (r - \sigma^2/2)\,h\,\frac{m+1}{2}$, $\sigma_G^2 = \sigma^2 h \frac{(m+1)(2m+1)}{6m}$, $h = T/m$,
  so its price is closed-form.

In [21]:
# Target: Asian arithmetic average call
def asian_payoff_from_paths(paths: np.ndarray, params: GBMParams) -> np.ndarray:
    A = paths.mean(axis=1)
    return np.maximum(A - params.K, 0.0)

# Control 1: EU call payoff from same paths
def eu_call_from_paths(paths: np.ndarray, params: GBMParams) -> np.ndarray:
    ST = paths[:, -1]
    return np.maximum(ST - params.K, 0.0)

# Control 2: geometric-average Asian call from same paths
def geo_asian_payoff_from_paths(paths: np.ndarray, params: GBMParams) -> np.ndarray:
    G = np.exp(np.log(paths).mean(axis=1))
    return np.maximum(G - params.K, 0.0)

def geo_asian_call_price(S0, K, r, sigma, T, m):
    """Discounted price of the discrete geometric Asian call (average over t_1..t_m)."""
    h = T / m
    mu_G = np.log(S0) + (r - 0.5 * sigma**2) * h * (m + 1) / 2
    sigma_G = sigma * np.sqrt(h * (m + 1) * (2 * m + 1) / (6 * m))
    d1 = (mu_G - np.log(K) + sigma_G**2) / sigma_G
    d2 = d1 - sigma_G
    return np.exp(-r * T) * (np.exp(mu_G + 0.5 * sigma_G**2) * norm.cdf(d1) - K * norm.cdf(d2))

### 7.3.3 Knock-In Option
Target: down-and-in call (discretely monitored). Control: EU call, true price = Black–Scholes.

In [22]:
# Target: Down-and-In Call
def barrier_di_payoff_from_paths(paths: np.ndarray, params: GBMParams, B: float) -> np.ndarray:
    S_min = paths.min(axis=1)
    ST = paths[:, -1]
    hit = (S_min <= B)
    return np.where(hit, np.maximum(ST - params.K, 0.0), 0.0)

# Control: EU call payoff from same paths (eu_call_from_paths above),
# true price bs_call_price(S0, K, r, sigma, T)

## 7.4 Importance Sampling
### 7.4.1 European Options

In [23]:
np.random.seed(42)

# Importance Sampling weight function
def Importance_Sampling_ratio(x, theta):
    return np.exp(-theta*x + 0.5*theta**2)

# Importance Sampling simulation for European call option pricing
def BS_Montecarlo_Important_Sampling(T, t, St, K, r, sigma, n: int):
    # T: Time of expiration
    # t: Current time
    # St: Underlying asset price at time t
    # K: Strike price
    # r: Risk-free interest rate (annualized)
    # sigma: Volatility of the asset's returns (annualized)
    # n: Number of Monte Carlo simulations

    theta = (np.log(K) - np.log(St) - (r - 0.5 * sigma**2) * (T - t)) / (sigma * np.sqrt(T - t))  # Importance sampling shift
    Z = np.random.standard_normal(n)  # Generate n standard normals
    Y = Z + theta
    ST = St * np.exp((r - sigma**2/2)*(T-t) + sigma*np.sqrt(T-t)*Y)  # Simulate asset prices at maturity
    price = np.exp(-r*(T-t))*np.sum(np.maximum(ST-K, 0)*Importance_Sampling_ratio(Y, theta))/n
    return price

### 7.4.2 Asian Options

In [24]:
np.random.seed(0)

def compute_theta(y, r, sigma, h, K, m, S0):
    """Compute the vector of shifts theta given y."""
    if y <= K:
        raise ValueError("y needs to be > K")
    theta = np.zeros(m)

    # Direct calculation of theta_1
    theta[0] = sigma * np.sqrt(h) * y / (y - K)
    S = [S0*np.exp((r - sigma**2/2)*(h) + sigma*np.sqrt(h)*theta[0])]

    # Recursive calculation of theta_{j+1}
    for j in range(m - 1):
        theta[j + 1] = theta[j] - sigma * np.sqrt(h) * S[-1] / (m * (y - K))
        S.append(S[-1]*np.exp((r - sigma**2/2)*(h) + sigma*np.sqrt(h)*theta[j+1]))
    return theta, S

def function_y(y, r, sigma, h, K, m, S0):
    """Function in y whose root we find to get optimal theta"""
    _, S = compute_theta(y, r, sigma, h, K, m, S0)
    S_mean = np.mean(S)
    return S_mean - y

def find_optimal_theta(r, sigma, h, K, m, S0, tol):
    """Find optimal theta with Brent's method on function in y"""
    lower_bound = K + 1e-4 * max(1.0, K)
    upper_bound = max(S0*np.exp(r*h*m), K)*10
    y_final = brentq(function_y, lower_bound, upper_bound, xtol=tol, rtol=tol, args=(r, sigma, h, K, m, S0))
    theta_final, _ = compute_theta(y_final, r, sigma, h, K, m, S0)
    return theta_final

def asian_IS(S0, K, r, sigma, T, n_steps, n_paths):
    # 1. Computing parameters
    dt = T / n_steps
    nudt = (r - 0.5 * sigma**2) * dt
    sdt = sigma * np.sqrt(dt)

    # 2. Computing the IS shift and likelihood ratios
    theta = find_optimal_theta(r, sigma, dt, K, n_steps, S0, tol=1e-8)
    Z = np.random.standard_normal(size=(n_paths, n_steps))
    Z += theta
    likelihood_ratio = np.exp(-Z @ theta + 0.5 * np.dot(theta, theta))

    # 3. Simulate paths and compute discounted payoffs
    log_increments = nudt + sdt * Z
    log_paths = np.cumsum(log_increments, axis=1)
    S = S0 * np.exp(log_paths)
    A = S.mean(axis=1)
    payoffs = np.maximum(A - K, 0.0) * likelihood_ratio
    disc_payoffs = np.exp(-r * T) * payoffs

    # 4. Compute statistics of the estimator
    variance = np.var(disc_payoffs, ddof=1)
    mean = np.mean(disc_payoffs)
    std_err = np.sqrt(variance / n_paths)
    conf_int = (mean - 1.96 * std_err, mean + 1.96 * std_err)

    return mean, std_err, variance, conf_int, n_paths

### 7.4.3 Digital Knock-In Options
`np.random.seed` called from Python does not seed numba's generator, so the seed is passed in and set inside the jitted function.

In [25]:
from numba import jit

@jit(nopython=True)
def knock_in_option_importance_sampling(S_0, K, H, T, r, sigma, m, n_sims, seed=42):
    np.random.seed(seed)

    dt = T / m
    sqrt_dt = np.sqrt(dt)

    # Distances (log-space)
    b = np.log(S_0 / H)  # Distance to Barrier (must be > 0)
    c = np.log(K / S_0)  # Distance from Start to Strike

    # theta calculation
    optimal_shift = (2*b + c) / (m * sigma**2 * dt)
    theta_down_x = (1/2 - (r/sigma**2)) - optimal_shift
    theta_up_x = (1/2 - (r/sigma**2)) + optimal_shift
    # Drifts for the Brownian Motion Z
    # Standard MC uses Z ~ N(0,1). We will use Z ~ N(shift, 1)
    shift_down = theta_down_x * sqrt_dt * sigma
    shift_up = theta_up_x * sqrt_dt * sigma

    payoffs = np.zeros(n_sims)

    for i in range(n_sims):

        # Initialize Path
        current_log_S = np.log(S_0)
        barrier_log = np.log(H)

        # Likelihood Ratio Accumulator (Sum of exponents)
        # LR = exp(-sum(theta * Z_biased - 0.5 * theta^2))
        log_lr = 0.0

        barrier_hit = False

        # Time Loop
        for t in range(m):

            if not barrier_hit:
                # Phase 1: Try to hit the barrier (Drift DOWN)
                # We sample from a shifted distribution
                Z_random = np.random.normal()
                Z_biased = Z_random + shift_down  # This is the Z we use for price

                # The correction term for shift 'theta' is: -theta * Z_biased + 0.5 * theta^2
                log_lr += -shift_down * Z_biased + 0.5 * (shift_down**2)

                # Evolve Price
                # d(ln S) = (r - 0.5*sigma^2) dt + sigma * sqrt(dt) * Z
                current_log_S += (r - 0.5 * sigma**2) * dt + sigma * sqrt_dt * Z_biased

                # Check Barrier
                if current_log_S <= barrier_log:
                    barrier_hit = True

            else:
                # Phase 2: Barrier hit, now go to Strike (Drift UP)
                Z_random = np.random.normal()
                Z_biased = Z_random + shift_up

                log_lr += -shift_up * Z_biased + 0.5 * (shift_up**2)

                current_log_S += (r - 0.5 * sigma**2) * dt + sigma * sqrt_dt * Z_biased

        # 3. Calculate Payoff & Weight
        S_T = np.exp(current_log_S)

        if barrier_hit and S_T > K:
            raw_payoff = np.exp(-r * T) * 10000.0
        else:
            raw_payoff = 0.0

        # Final Weight
        likelihood_ratio = np.exp(log_lr)

        # Weighted Payoff (Unbiased Estimator)
        payoffs[i] = raw_payoff * likelihood_ratio

    # 4. Statistics (numba's np.var has no ddof, so rescale to the unbiased estimate)
    price_estimate = np.mean(payoffs)
    variance_is = np.var(payoffs) * n_sims / (n_sims - 1)
    se = np.sqrt(variance_is / n_sims)
    z = 1.96
    ci = (price_estimate - z*se, price_estimate + z*se)

    return price_estimate, variance_is

## Sanity checks
Quick runs of every function at moderate N to confirm the code works. These are not the report's experiments.

In [26]:
from IPython.display import display

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

def row(method, price, se, variance, n, benchmark=np.nan):
    return {"Method": method, "Price": float(price), "Std. error": float(se),
            "CI low": float(price) - 1.96 * float(se), "CI high": float(price) + 1.96 * float(se),
            "Variance": float(variance), "N": int(n), "Benchmark": benchmark}

def row_mc(method, res: MCResult, benchmark=np.nan):
    return row(method, res.price, res.std_error, res.variance, res.n_paths, benchmark)

def show(rows):
    display(pd.DataFrame(rows).set_index("Method").dropna(axis=1, how="all"))

S0, K, r, sigma, T = 100.0, 100.0, 0.05, 0.2, 1.0
rng = np.random.default_rng(1)

### European call: plain MC vs antithetic
S0 = K = 100, r = 0.05, σ = 0.2, T = 1, N = 200,000. Benchmark: Black–Scholes.

In [27]:
bs = bs_call_price(S0, K, r, sigma, T)
show([
    row_mc("Plain MC", mc_euro_bs_plain(S0, K, r, sigma, T, 200_000, rng=rng), bs),
    row_mc("Antithetic", mc_euro_bs_antithetic(S0, K, r, sigma, T, 200_000, rng=rng), bs),
])

,Price,Std. error,CI low,CI high,Variance,N,Benchmark
Method,,,,,,,
Plain MC,10.3993,0.0328,10.3349,10.4636,215.6475,200000,10.4506
Antithetic,10.4357,0.0232,10.3903,10.4812,53.7965,200000,10.4506


### Arithmetic Asian call: plain MC, antithetic, importance sampling
Same parameters. Plain and antithetic use 50 steps (S0 included in the average); IS uses 150 steps (S0 excluded), N = 50,000.

In [28]:
is_mean, is_se, is_var, _, is_n = asian_IS(S0, K, r, sigma, T, 150, 50_000)
show([
    row_mc("Plain MC (m=50)", asian_arith_plain(S0, K, r, sigma, T, 50, 100_000, rng=rng)),
    row_mc("Antithetic (m=50)", asian_arith_antithetic(S0, K, r, sigma, T, 50, 100_000, rng=rng)),
    row("Importance sampling (m=150)", is_mean, is_se, is_var, is_n),
])

,Price,Std. error,CI low,CI high,Variance,N
Method,,,,,,
Plain MC (m=50),5.7983,0.0252,5.7489,5.8478,63.6842,100000
Antithetic (m=50),5.7314,0.0173,5.6975,5.7653,14.9837,100000
Importance sampling (m=150),5.7955,0.0125,5.7710,5.8201,7.8537,50000


### Deep OTM European call: importance sampling
K = 140, N = 100,000.

In [29]:
pd.DataFrame([{
    "IS price": BS_Montecarlo_Important_Sampling(1, 0, 100, 140, 0.05, 0.2, 100_000),
    "Black–Scholes": bs_call_price(100, 140, 0.05, 0.2, 1),
}])

,IS price,Black–Scholes
0,0.7854,0.7850


### Digital knock-in: plain MC vs importance sampling
S0 = 95, K = 100, H = 85, r = 0.05, σ = 0.15, T = 1, m = 50, N = 100,000.

In [30]:
N_ki = 100_000
p_mc, v_mc = standard_mc_knock_in(95, 100, 85, 1.0, 0.05, 0.15, 50, N_ki)
p_is, v_is = knock_in_option_importance_sampling(95.0, 100.0, 85.0, 1.0, 0.05, 0.15, 50, N_ki)
show([
    row("Plain MC", p_mc, np.sqrt(v_mc / N_ki), v_mc, N_ki),
    row("Importance sampling", p_is, np.sqrt(v_is / N_ki), v_is, N_ki),
])
print(f"VRF (IS vs plain): {v_mc / v_is:.1f}")

,Price,Std. error,CI low,CI high,Variance,N
Method,,,,,,
Plain MC,263.1101,4.9331,253.4411,272.7790,"2,433,577.7310",100000
Importance sampling,259.8816,1.2900,257.3533,262.4099,"166,397.4610",100000


VRF (IS vs plain): 14.6


### Control variates
GBMParams defaults (r = 0.03), N = 100,000, m = 50 for path-dependent options. VRF = Var(plain) / Var(CV).

In [31]:
p = GBMParams(n=100_000)
m = 50
bs_p = bs_call_price(p.S0, p.K, p.r, p.sigma, p.T)
cv_cases = {
    "EU call | control S_T":            (eu_call_payoff_from_paths, eu_call_control_from_paths, eu_call_control_true_price(p), 1),
    "Asian | control EU call":          (asian_payoff_from_paths, eu_call_from_paths, bs_p, m),
    "Asian | control geometric Asian":  (asian_payoff_from_paths, geo_asian_payoff_from_paths, geo_asian_call_price(p.S0, p.K, p.r, p.sigma, p.T, m), m),
    "Knock-in B=85 | control EU call":  (lambda x, q: barrier_di_payoff_from_paths(x, q, 85.0), eu_call_from_paths, bs_p, m),
}
rows = []
for name, (tgt, ctl, true_price, steps) in cv_cases.items():
    mu_mc, var_mc, se_mc, _ = mc_price_generic_paths(p, tgt, n_steps=steps)
    mu_cv, var_cv, se_cv, _, b_hat, rho_hat = mc_price_cv_paths(p, tgt, ctl, true_price, n_steps=steps)
    rows.append({"Case": name, "Price MC": mu_mc, "Price CV": mu_cv, "SE MC": se_mc, "SE CV": se_cv,
                 "ρ̂": rho_hat, "b̂": b_hat, "VRF": var_mc / var_cv})
display(pd.DataFrame(rows).set_index("Case"))
print(f"Black–Scholes (r = 0.03): {bs_p:.4f}")

,Price MC,Price CV,SE MC,SE CV,ρ̂,b̂,VRF
Case,,,,,,,
EU call | control S_T,9.3709,9.3928,0.0444,0.0183,0.9111,0.6355,5.8869
Asian | control EU call,5.3504,5.3541,0.0248,0.0135,0.8398,0.4679,3.3922
Asian | control geometric Asian,5.3504,5.3671,0.0248,0.0007,0.9996,1.0334,"1,411.4211"
Knock-in B=85 | control EU call,0.3215,0.3216,0.0072,0.0071,0.0693,0.0112,1.0048


Black–Scholes (r = 0.03): 9.4134


### Geometric Asian: closed form vs MC

In [32]:
pd.DataFrame([{
    "Closed form": geo_asian_call_price(p.S0, p.K, p.r, p.sigma, p.T, m),
    "MC estimate": mc_price_generic_paths(p, geo_asian_payoff_from_paths, n_steps=m)[0],
    "MC std. error": mc_price_generic_paths(p, geo_asian_payoff_from_paths, n_steps=m)[2],
}])

,Closed form,MC estimate,MC std. error
0,5.1701,5.1539,0.0240


### `collect_convergence_data_paths` with and without `B`

In [33]:
cols = ["n", "price_mc", "price_cv", "se_mc", "se_cv", "rho_hat"]
res = collect_convergence_data_paths("EU", p, eu_call_payoff_from_paths, eu_call_control_from_paths, p.S0,
                                     [1_000, 10_000, 100_000], n_steps=1)
res_b = collect_convergence_data_paths("KI", p, barrier_di_payoff_from_paths, eu_call_from_paths, bs_p,
                                       [1_000, 10_000, 100_000], n_steps=m, B=85.0)
print("European call, control S_T")
display(pd.DataFrame(res)[cols].set_index("n"))
print("Down-and-in call (B = 85), control EU call")
display(pd.DataFrame(res_b)[cols].set_index("n"))

European call, control S_T


,price_mc,price_cv,se_mc,se_cv,rho_hat
n,,,,,
1000,7.8923,8.8966,0.3997,0.1773,0.8962
10000,9.1974,9.3669,0.1394,0.0579,0.9097
100000,9.3709,9.3928,0.0444,0.0183,0.9111


Down-and-in call (B = 85), control EU call


,price_mc,price_cv,se_mc,se_cv,rho_hat
n,,,,,
1000,0.4076,0.4094,0.0763,0.0762,0.0655
10000,0.3042,0.3030,0.0209,0.0209,0.0526
100000,0.3215,0.3216,0.0072,0.0071,0.0693
